# Data Pipeline Exploration

Explore CIFAR-10, Fashion-MNIST, and CIFAR-100 data loaders and basic statistics.

In [1]:
# Colab bootstrap cell
!git clone https://github.com/kar137/ouroboros.git
%cd ouroboros

Cloning into 'ouroboros'...
remote: Enumerating objects: 89, done.
remote: Counting objects: 100% (89/89), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 89 (delta 46), reused 60 (delta 21), pack-reused 0 (from 0)
Receiving objects: 100% (89/89), 405.59 KiB | 2.88 MiB/s, done.
Resolving deltas: 100% (46/46), done.
/content/ouroboros


In [2]:
import sys
from pathlib import Path

candidate_roots = [
    Path('/content/ouroboros'),
]
project_root = next((p for p in candidate_roots if p.exists()), None)
if project_root is None:
    raise FileNotFoundError('Project root not found. Update candidate_roots.')
sys.path.insert(0, str(project_root))

import torch
import matplotlib.pyplot as plt
from src.data_loaders import get_cifar10_loaders, get_fashion_mnist_loaders, get_cifar100_loaders
from src.utils import set_seed, ensure_dirs

set_seed(42)
ensure_dirs('results/figures')

loaders = {
    'CIFAR-10': get_cifar10_loaders,
    'Fashion-MNIST': get_fashion_mnist_loaders,
    'CIFAR-100': get_cifar100_loaders,
}

batch_size = 64
num_workers = 2
data_dir = 'assets'

for name, fn in loaders.items():
    train_loader, _ = fn(batch_size, num_workers, data_dir)
    inputs, targets = next(iter(train_loader))
    print(name, 'batch shape:', inputs.shape, 'targets:', targets.shape)
    print('mean:', inputs.mean(dim=(0, 2, 3)).tolist())
    print('std:', inputs.std(dim=(0, 2, 3)).tolist())

    imgs = inputs[:8]
    if imgs.shape[1] == 1:
        imgs = imgs.repeat(1, 3, 1, 1)
    grid = torch.clamp(imgs * 0.5 + 0.5, 0, 1)
    fig, axes = plt.subplots(1, 8, figsize=(12, 2))
    for i, ax in enumerate(axes):
        ax.imshow(grid[i].permute(1, 2, 0))
        ax.axis('off')
    plt.suptitle(name)
    fig_path = Path('results/figures') / f"{name.lower().replace(' ', '_')}_samples.png"
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print('Saved sample grid to', fig_path)

100%|██████████| 170M/170M [00:10<00:00, 15.9MB/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


CIFAR-10 batch shape: torch.Size([64, 3, 32, 32]) targets: torch.Size([64])
mean: [-0.2476944476366043, -0.2555772364139557, -0.25725385546684265]
std: [1.1498056650161743, 1.164650559425354, 1.0830128192901611]
Saved sample grid to results/figures/cifar-10_samples.png


100%|██████████| 26.4M/26.4M [00:03<00:00, 7.82MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 142kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 2.69MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 19.3MB/s]


Fashion-MNIST batch shape: torch.Size([64, 1, 28, 28]) targets: torch.Size([64])
mean: [-0.141872838139534]
std: [0.9269124865531921]
Saved sample grid to results/figures/fashion-mnist_samples.png


100%|██████████| 169M/169M [00:10<00:00, 16.2MB/s]


CIFAR-100 batch shape: torch.Size([64, 3, 32, 32]) targets: torch.Size([64])
mean: [-0.101308174431324, -0.166782945394516, -0.18988031148910522]
std: [1.260703206062317, 1.2344316244125366, 1.1646766662597656]
Saved sample grid to results/figures/cifar-100_samples.png
